In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'cardekho-used-car-data' dataset.
Path to dataset files: /kaggle/input/cardekho-used-car-data


In [8]:
import pandas as pd
import os

# List contents of the downloaded directory to find the correct file name
# print(os.listdir(path))

# Once you identify the correct file name, replace 'your_actual_file_name.csv' below
df = pd.read_csv(path + "/cardekho_dataset.csv")
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


#Task 1:

In [9]:
df.shape

(15411, 14)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), object(6)
memory usage: 1.6+ MB


In [11]:
df.isnull().sum() * 100 / len(df)

,0
Unnamed: 0,0.0
car_name,0.0
brand,0.0
model,0.0
vehicle_age,0.0
km_driven,0.0
seller_type,0.0
fuel_type,0.0
transmission_type,0.0
mileage,0.0


In [19]:
#Dropping selling price missing values

df_selling = df.dropna(subset = ['selling_price'], inplace=True)
print(df_selling)

None


In [20]:
#Clean mileage, engine, max_power
cols_tofix = ['mileage', 'engine', 'max_power']

for col in cols_tofix:
  #strip to  extract
  df[col] = df[col].astype(str).str.extract(r'(\d+\.?\d*)')[0]

  #Convert too numeric
  df[col] = pd.to_numeric(df[col], errors ='coerce')

  #Fill remaining nulls with the column's median
  df[col] = df[col].fillna(df[col].median())

In [21]:
#remove rows where selling price is 999,999,999 or below 10,000
df = df[(df['selling_price'] != 999999999) & (df['selling_price'] >= 10000)]

In [22]:
#Drop all duplicates
df.drop_duplicates(inplace=True)

In [24]:
df.shape

(15411, 14)

#Task 2:

In [25]:
# 1. Label Encoding for transmission_type
#Mapping Manual to 0 and Automatic to 1

df['transmission_type'] = df['transmission_type'].map({"Manual":0, "Automatic": 1 })

In [27]:
#One-Hot Encoding for fuel_type and seller_type
df = pd.get_dummies(df, columns=['fuel_type', 'seller_type'], drop_first=True)

In [28]:
print(df.columns.tolist())

['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price', 'fuel_type_Diesel', 'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer']


**Understanding drop_first=True** The primary reason to use drop_first=True is to avoid Multicollinearity(Specifically the "Dummy variable Trap"). In statistical models(like Linear Regression), if you include all categories as columns, the variables become perfectly predictable from one another.


In [29]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import numpy as np

In [31]:
#1. define X and y
X = df.drop(columns=['selling_price'])
y = df['selling_price']

In [32]:
#2. Drop any remaining non-numeric columns from x (like 'name' or 'torque)
X = X.select_dtypes(include=[np.number])

In [35]:
#3. Split into 80% train and 20% test.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [36]:
#4.Compute the Baseline MAE
#We predict the average training price for every car in the test set
baseline_preds = np.full(shape=y_test.shape, fill_value=y_train.mean())
baseline_mae = mean_absolute_error(y_test, baseline_preds)

print(f"Baseline MAE: {round(baseline_mae)}")

Baseline MAE: 468748
